In [1]:
import osmnx as ox
import networkx as nx

# 1. Tải mạng lưới đường bộ của Quận 1 (network_type="drive" lấy đường xe chạy được, bỏ lối đi bộ trong công viên)
place_name = "District 1, Ho Chi Minh City, Vietnam"
G_raw = ox.graph_from_place(place_name, network_type="drive")

print(f"Số nút (ngã rẽ) ban đầu: {G_raw.number_of_nodes()}")
print(f"Số cạnh (đoạn đường) ban đầu: {G_raw.number_of_edges()}")

# 2. Giữ lại thành phần liên thông mạnh lớn nhất (Largest Strongly Connected Component)
# Lý do: Đồ thị đường phố là đồ thị có hướng (MultiDiGraph) vì có đường 1 chiều.
# Giữ thành phần liên thông mạnh đảm bảo từ bất kỳ nút nào cũng đi tới được mọi nút khác mà không bị kẹt ngõ cụt!
G = ox.truncate.largest_component(G_raw, strongly=True)

print(f"Số nút sau khi làm sạch: {G.number_of_nodes()}")
print(f"Số cạnh sau khi làm sạch: {G.number_of_edges()}")

Số nút (ngã rẽ) ban đầu: 958
Số cạnh (đoạn đường) ban đầu: 2022
Số nút sau khi làm sạch: 916
Số cạnh sau khi làm sạch: 1957


In [2]:
# Lấy thử 1 nút đầu tiên để xem thông tin
sample_node_id, sample_node_data = list(G.nodes(data=True))[0]
print("Dữ liệu 1 nút (ngã tư):", sample_node_id, sample_node_data)

# Lấy thử 1 cạnh đầu tiên để xem thông tin
u, v, key, sample_edge_data = list(G.edges(keys=True, data=True))[0]
print(f"\nDữ liệu 1 đoạn đường ({u} -> {v}):")
for k, val in sample_edge_data.items():
    print(f"  - {k}: {val}")

Dữ liệu 1 nút (ngã tư): 366440881 {'y': 10.7926816, 'x': 106.6958961, 'street_count': 4}

Dữ liệu 1 đoạn đường (366440881 -> 6637081019):
  - osmid: 817909613
  - highway: tertiary
  - oneway: False
  - reversed: True
  - length: 10.611311922279608


In [3]:
# Bảng vận tốc trung bình xe máy tại TP.HCM (đơn vị: km/h)
MOTO_SPEED_KMH = {
    "trunk": 35.0,
    "trunk_link": 25.0,
    "primary": 32.0,
    "primary_link": 22.0,
    "secondary": 30.0,
    "secondary_link": 20.0,
    "tertiary": 28.0,
    "tertiary_link": 20.0,
    "residential": 22.0,
    "unclassified": 22.0,
    "living_street": 15.0,
    "service": 15.0,
}
DEFAULT_SPEED_KMH = 20.0

def gan_chi_phi_xe_may(G):
    for u, v, key, data in G.edges(keys=True, data=True):
        # 1. Xử lý loại đường (OSM có thể trả về str hoặc list)
        hw = data.get("highway", "")
        if isinstance(hw, list):
            hw = hw[0]  # Lấy loại đường đầu tiên
            
        speed_kmh = MOTO_SPEED_KMH.get(hw, DEFAULT_SPEED_KMH)
        speed_ms = speed_kmh * 1000 / 3600  # Đổi km/h sang m/s
        
        # 2. Thời gian chạy thuần túy (giây)
        length_m = data.get("length", 0.0)
        run_time = length_m / speed_ms
        
        # 3. Phạt qua nút giao tại đỉnh đích v
        street_count = G.nodes[v].get("street_count", 2)
        if street_count >= 4:
            penalty = 8.0   # Ngã tư / đèn đỏ
        elif street_count == 3:
            penalty = 3.0   # Ngã ba
        else:
            penalty = 0.0
            
        # Lưu thuộc tính travel_time (giây) vào cạnh
        data["travel_time"] = round(run_time + penalty, 2)
        
    return G

# Áp dụng mô hình chi phí lên đồ thị Quận 1
G = gan_chi_phi_xe_may(G)

# Kiểm tra lại đoạn đường mẫu lúc nãy
u, v, key, sample_edge = list(G.edges(keys=True, data=True))[0]
print(f"Đoạn đường ({u} -> {v}): dài {sample_edge['length']:.1f}m | loại: {sample_edge['highway']} | thời gian xe máy: {sample_edge['travel_time']} giây")

# Lưu đồ thị ra file .graphml trong thư mục data/
output_path = "data/q1_moto.graphml"
ox.save_graphml(G, filepath=output_path)
print(f"Đã xuất đồ thị thành công tại: {output_path}")

Đoạn đường (366440881 -> 6637081019): dài 10.6m | loại: tertiary | thời gian xe máy: 9.36 giây
Đã xuất đồ thị thành công tại: data/q1_moto.graphml
